<a href="https://colab.research.google.com/github/Rusiru49/J26-DS-315_Aero-Optimization-EV-Energy/blob/IT23278080_Component01/SLDC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import glob
import warnings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

In [ ]:
# ============================================================
# 1. CONFIGURATION
# ============================================================

DATA_FOLDER = "data"

OUTPUT_FOLDER = "sldc_analysis"

RESAMPLE_RATE = "1s"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

In [ ]:
# ============================================================
# 2. LOAD CSV FILES
# ============================================================

files = glob.glob(os.path.join(DATA_FOLDER, "*.csv"))

if len(files) == 0:
    raise FileNotFoundError(
        "No CSV files found. Put your CSV files inside the 'data' folder."
    )

print(f"Found {len(files)} CSV files")

Found 2 CSV files


In [ ]:
# ============================================================
# 3. FUNCTION TO STANDARDIZE COLUMN NAMES
# ============================================================

def standardize_columns(df):

    rename_map = {

        "time": "time",

        "Vehicle speed (km/h)": "speed",

        "Vehicle acceleration (g)": "acceleration_g",

        "Engine RPM (rpm)": "rpm",

        "Throttle position (%)": "throttle",

        "Calculated engine load value (%)": "engine_load",

        "Calculated instant fuel rate (L/h)": "fuel_rate",

        "Calculated instant fuel consumption (L/100km)":
            "fuel_consumption",

        "Distance travelled (km)": "distance",

        "Engine coolant temperature (℃)":
            "coolant_temp",

        "Intake air temperature (℃)":
            "intake_temp",

        "Intake manifold absolute pressure (kPa)":
            "intake_pressure",

        "OBD Module Voltage (V)":
            "obd_voltage",

        "Timing advance (°)":
            "timing_advance",

        "Long term fuel % trim - Bank 1 (%)":
            "long_fuel_trim",

        "Short term fuel % trim - Bank 1 (%)":
            "short_fuel_trim",

        "Calculated boost (bar)":
            "boost",

        "Barometric pressure (kPa)":
            "barometric_pressure",

        "Instant engine power (based on fuel consumption) (hp)":
            "engine_power"

    }

    df = df.rename(columns=rename_map)

    return df

In [ ]:
# ============================================================
# 4. CLEAN INDIVIDUAL FILE
# ============================================================

def clean_data(file):

    print("\nProcessing:", os.path.basename(file))

    df = pd.read_csv(file)

    # Remove completely empty columns
    df = df.dropna(axis=1, how="all")

    # Standardize names
    df = standardize_columns(df)

    # Convert timestamp
    df["time"] = pd.to_timedelta(df["time"])

    # Sort
    df = df.sort_values("time")

    # Remove duplicate timestamps
    df = df.drop_duplicates(subset="time")

    # Numeric columns
    numeric_columns = [
        c for c in df.columns
        if c != "time"
    ]

    for c in numeric_columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    return df

In [ ]:
# ============================================================
# 5. RESAMPLE TO 1 SECOND
# ============================================================

def resample_data(df):

    df = df.copy()

    # Convert timedelta to seconds
    df["seconds"] = df["time"].dt.total_seconds()

    # Create datetime-like index
    df["timestamp"] = pd.to_datetime(
        df["seconds"],
        unit="s"
    )

    df = df.set_index("timestamp")

    # Numeric columns
    numeric_columns = df.select_dtypes(
        include=np.number
    ).columns

    # Resample
    df = df[numeric_columns].resample(
        RESAMPLE_RATE
    ).mean()

    # Interpolate only short gaps
    df = df.interpolate(
        method="linear",
        limit=3
    )

    return df.reset_index(drop=True)

In [ ]:
# ============================================================
# 6. CALCULATE ACCELERATION FROM SPEED
# ============================================================

def calculate_acceleration(df):

    if "speed" not in df.columns:
        return df

    # Speed km/h → m/s
    speed_ms = df["speed"] / 3.6

    # Since data is 1 second
    df["calculated_acceleration_ms2"] = speed_ms.diff()

    # Convert to g
    df["calculated_acceleration_g"] = (
        df["calculated_acceleration_ms2"] / 9.80665
    )

    return df

In [ ]:
# ============================================================
# 7. LIMIT PHYSICALLY UNREALISTIC VALUES
# ============================================================

def remove_outliers(df):

    # Vehicle speed
    if "speed" in df.columns:

        df.loc[
            (df["speed"] < 0) |
            (df["speed"] > 160),
            "speed"
        ] = np.nan

    # Acceleration
    if "calculated_acceleration_g" in df.columns:

        df.loc[
            (df["calculated_acceleration_g"] < -0.5) |
            (df["calculated_acceleration_g"] > 0.5),
            "calculated_acceleration_g"
        ] = np.nan

    return df

In [ ]:
# ============================================================
# 8. CLASSIFY DRIVING STATE
# ============================================================

def classify_driving(df):

    speed = df["speed"]

    acceleration = df[
        "calculated_acceleration_g"
    ]

    conditions = [

        speed <= 1,

        (acceleration > 0.01),

        (acceleration < -0.01),

        (abs(acceleration) <= 0.01) &
        (speed > 1)

    ]

    states = [

        "STOP",

        "ACCELERATION",

        "DECELERATION",

        "CRUISE"

    ]

    df["driving_state"] = np.select(
        conditions,
        states,
        default="UNKNOWN"
    )

    return df

In [ ]:
# ============================================================
# 9. CALCULATE TRIP STATISTICS
# ============================================================

def calculate_statistics(df, filename):

    stats = {}

    stats["file"] = filename

    stats["duration_min"] = len(df) / 60

    if "speed" in df.columns:

        stats["average_speed_kmh"] = df[
            "speed"
        ].mean()

        stats["maximum_speed_kmh"] = df[
            "speed"
        ].max()

        stats["speed_std"] = df[
            "speed"
        ].std()

    if "calculated_acceleration_g" in df.columns:

        stats["average_acceleration_g"] = df[
            "calculated_acceleration_g"
        ].mean()

        stats["maximum_acceleration_g"] = df[
            "calculated_acceleration_g"
        ].max()

        stats["maximum_deceleration_g"] = df[
            "calculated_acceleration_g"
        ].min()

    if "distance" in df.columns:

        stats["distance_km"] = df[
            "distance"
        ].max()

    # Driving-state percentages
    if "driving_state" in df.columns:

        percentages = (
            df["driving_state"]
            .value_counts(normalize=True)
            * 100
        )

        stats["stop_percent"] = percentages.get(
            "STOP", 0
        )

        stats["acceleration_percent"] = percentages.get(
            "ACCELERATION", 0
        )

        stats["deceleration_percent"] = percentages.get(
            "DECELERATION", 0
        )

        stats["cruise_percent"] = percentages.get(
            "CRUISE", 0
        )

    return stats

In [ ]:
# ============================================================
# 10. PLOT SPEED PROFILE
# ============================================================

def plot_speed(df, filename):

    plt.figure(figsize=(14, 5))

    plt.plot(
        df["speed"],
        linewidth=1
    )

    plt.xlabel("Time (seconds)")

    plt.ylabel("Vehicle Speed (km/h)")

    plt.title(
        f"SLDC Speed Profile - {filename}"
    )

    plt.grid(True)

    plt.tight_layout()

    output = os.path.join(
        OUTPUT_FOLDER,
        filename.replace(".csv", "_speed.png")
    )

    plt.savefig(output, dpi=300)

    plt.close()

In [ ]:
# ============================================================
# 11. PLOT ACCELERATION
# ============================================================

def plot_acceleration(df, filename):

    plt.figure(figsize=(14, 5))

    plt.plot(
        df["calculated_acceleration_g"],
        linewidth=1
    )

    plt.axhline(
        0,
        linestyle="--"
    )

    plt.xlabel("Time (seconds)")

    plt.ylabel("Acceleration (g)")

    plt.title(
        f"Acceleration Profile - {filename}"
    )

    plt.grid(True)

    plt.tight_layout()

    output = os.path.join(
        OUTPUT_FOLDER,
        filename.replace(".csv", "_acceleration.png")
    )

    plt.savefig(output, dpi=300)

    plt.close()

In [ ]:
# ============================================================
# 12. SPEED DISTRIBUTION
# ============================================================

def plot_speed_distribution(df, filename):

    plt.figure(figsize=(10, 5))

    plt.hist(
        df["speed"].dropna(),
        bins=20
    )

    plt.xlabel("Vehicle Speed (km/h)")

    plt.ylabel("Frequency")

    plt.title(
        f"Speed Distribution - {filename}"
    )

    plt.grid(True)

    plt.tight_layout()

    output = os.path.join(
        OUTPUT_FOLDER,
        filename.replace(".csv", "_speed_distribution.png")
    )

    plt.savefig(output, dpi=300)

    plt.close()

In [ ]:
# ============================================================
# 13. DRIVING STATE DISTRIBUTION
# ============================================================

def plot_driving_states(df, filename):

    percentages = (
        df["driving_state"]
        .value_counts(normalize=True)
        * 100
    )

    plt.figure(figsize=(8, 5))

    percentages.plot(
        kind="bar"
    )

    plt.xlabel("Driving State")

    plt.ylabel("Percentage (%)")

    plt.title(
        f"Driving State Distribution - {filename}"
    )

    plt.tight_layout()

    output = os.path.join(
        OUTPUT_FOLDER,
        filename.replace(".csv", "_states.png")
    )

    plt.savefig(output, dpi=300)

    plt.close()

In [ ]:
# ============================================================
# 14. PROCESS ALL FILES
# ============================================================

all_statistics = []

processed_data = []

for file in files:

    filename = os.path.basename(file)

    try:

        df = clean_data(file)

        df = resample_data(df)

        df = calculate_acceleration(df)

        df = remove_outliers(df)

        df = classify_driving(df)

        # Statistics
        stats = calculate_statistics(
            df,
            filename
        )

        all_statistics.append(stats)

        processed_data.append(
            df.assign(
                source_file=filename
            )
        )

        # Plots
        plot_speed(
            df,
            filename
        )

        plot_acceleration(
            df,
            filename
        )

        plot_speed_distribution(
            df,
            filename
        )

        plot_driving_states(
            df,
            filename
        )

        print(
            f"Completed: {filename}"
        )

    except Exception as e:

        print(
            f"ERROR processing {filename}: {e}"
        )


Processing: 2026-08-23 10-44-49_Galeni to Kottawa.csv
Completed: 2026-08-23 10-44-49_Galeni to Kottawa.csv

Processing: 2026-08-23 16-33-45_Kottawa to Galeni.csv
Completed: 2026-08-23 16-33-45_Kottawa to Galeni.csv


In [ ]:
# ============================================================
# 15. CREATE SUMMARY TABLE
# ============================================================

summary = pd.DataFrame(
    all_statistics
)

summary.to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "trip_summary.csv"
    ),
    index=False
)

print("\n==============================")
print("TRIP SUMMARY")
print("==============================")

print(
    summary.round(3)
)


TRIP SUMMARY
                                        file  duration_min  average_speed_kmh  \
0  2026-08-23 10-44-49_Galeni to Kottawa.csv        14.783             60.056   
1  2026-08-23 16-33-45_Kottawa to Galeni.csv        14.267             63.764   

   maximum_speed_kmh  speed_std  average_acceleration_g  \
0               88.0     18.170                   0.002   
1               94.0     22.417                   0.001   

   maximum_acceleration_g  maximum_deceleration_g  distance_km  stop_percent  \
0                   0.113                  -0.057       14.227         5.975   
1                   0.198                  -0.227       14.707         5.958   

   acceleration_percent  deceleration_percent  cruise_percent  
0                19.278                14.769          56.144  
1                23.715                19.042          48.364  


In [ ]:
# ============================================================
# 16. COMBINE ALL TRIPS
# ============================================================

if len(processed_data) > 0:

    combined = pd.concat(
        processed_data,
        ignore_index=True
    )

    combined.to_csv(
        os.path.join(
            OUTPUT_FOLDER,
            "combined_sldc_dataset.csv"
        ),
        index=False
    )

    print(
        "\nCombined dataset created."
    )


Combined dataset created.


In [ ]:
# ============================================================
# 17. OVERALL SPEED DISTRIBUTION
# ============================================================

if len(processed_data) > 0:

    plt.figure(figsize=(10, 5))

    plt.hist(
        combined["speed"].dropna(),
        bins=30
    )

    plt.xlabel(
        "Vehicle Speed (km/h)"
    )

    plt.ylabel(
        "Frequency"
    )

    plt.title(
        "Combined SLDC Speed Distribution"
    )

    plt.grid(True)

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            OUTPUT_FOLDER,
            "combined_speed_distribution.png"
        ),
        dpi=300
    )

    plt.close()

In [ ]:
# ============================================================
# 18. CORRELATION ANALYSIS
# ============================================================

if len(processed_data) > 0:

    important_columns = [

        "speed",
        "calculated_acceleration_g",
        "rpm",
        "throttle",
        "engine_load",
        "fuel_rate",
        "fuel_consumption",
        "engine_power"

    ]

    existing_columns = [
        c for c in important_columns
        if c in combined.columns
    ]

    correlation = combined[
        existing_columns
    ].corr()

    correlation.to_csv(
        os.path.join(
            OUTPUT_FOLDER,
            "correlation_matrix.csv"
        )
    )

    print("\nCorrelation Matrix")

    print(
        correlation.round(3)
    )


print("\n==============================")
print("SLDC ANALYSIS COMPLETE")
print("==============================")


Correlation Matrix
                           speed  calculated_acceleration_g    rpm  throttle  \
speed                      1.000                     -0.108  0.255     0.335   
calculated_acceleration_g -0.108                      1.000  0.306     0.302   
rpm                        0.255                      0.306  1.000     0.904   
throttle                   0.335                      0.302  0.904     1.000   
engine_load                  NaN                        NaN    NaN       NaN   
fuel_rate                  0.323                      0.301  0.968     0.953   
fuel_consumption          -0.234                      0.203  0.259     0.233   
engine_power               0.323                      0.301  0.968     0.953   

                           engine_load  fuel_rate  fuel_consumption  \
speed                              NaN      0.323            -0.234   
calculated_acceleration_g          NaN      0.301             0.203   
rpm                                NaN      0.

In [ ]:
pd.read_csv('/content/sldc_analysis/combined_sldc_dataset.csv')

,time,Average fuel consumption (L/100km),Average fuel consumption (Today) (L/100km),Average fuel consumption (total) (L/100km),Average fuel consumption (Week) (L/100km),Average fuel consumption 10 sec (L/100km),Average speed (km/h),barometric_pressure,boost,engine_load,...,short_fuel_trim,throttle,timing_advance,acceleration_g,speed,seconds,calculated_acceleration_ms2,calculated_acceleration_g,driving_state,source_file
0,0 days 10:45:34.551666666,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.013250,NaN,...,NaN,NaN,NaN,NaN,0.0,38734.551667,NaN,NaN,STOP,2026-08-23 10-44-49_Galeni to Kottawa.csv
1,0 days 10:45:35.221500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.012438,NaN,...,NaN,16.470588,NaN,NaN,0.0,38735.221500,0.000000,0.000000,STOP,2026-08-23 10-44-49_Galeni to Kottawa.csv
2,0 days 10:45:36.263722222,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.011625,NaN,...,NaN,16.470588,NaN,NaN,0.0,38736.263722,0.000000,0.000000,STOP,2026-08-23 10-44-49_Galeni to Kottawa.csv
3,0 days 10:45:37.305944444,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.010812,NaN,...,NaN,16.470588,NaN,NaN,0.0,38737.305944,0.000000,0.000000,STOP,2026-08-23 10-44-49_Galeni to Kottawa.csv
4,0 days 10:45:38.348166666,NaN,NaN,NaN,NaN,NaN,NaN,101.0,-0.010000,NaN,...,NaN,16.470588,NaN,NaN,NaN,38738.348167,NaN,NaN,UNKNOWN,2026-08-23 10-44-49_Galeni to Kottawa.csv
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1738,0 days 16:48:32.545666666,3.937311,4.149675,5.286609,4.149675,3.913804,63.969686,NaN,-0.180000,NaN,...,NaN,23.921569,NaN,0.077529,38.0,60512.545667,0.277778,0.028325,ACCELERATION,2026-08-23 16-33-45_Kottawa to Galeni.csv
1739,0 days 16:48:33.530750,3.939876,4.150926,5.287413,4.150926,3.916359,63.928304,NaN,-0.220000,NaN,...,NaN,20.784314,NaN,0.057039,39.0,60513.530750,0.277778,0.028325,ACCELERATION,2026-08-23 16-33-45_Kottawa to Galeni.csv
1740,0 days 16:48:34.462000,3.935892,4.148794,5.285169,4.148794,3.920580,63.908250,NaN,-0.550000,NaN,...,NaN,18.627451,NaN,0.046745,39.5,60514.462000,0.138889,0.014163,ACCELERATION,2026-08-23 16-33-45_Kottawa to Galeni.csv
1741,0 days 16:48:35.412750,3.933827,4.147688,5.284004,4.147688,3.922416,63.888197,NaN,0.000000,NaN,...,NaN,16.470588,NaN,0.036451,40.0,60515.412750,0.138889,0.014163,ACCELERATION,2026-08-23 16-33-45_Kottawa to Galeni.csv


In [ ]:
pd.read_csv('/content/sldc_analysis/trip_summary.csv')

,file,duration_min,average_speed_kmh,maximum_speed_kmh,speed_std,average_acceleration_g,maximum_acceleration_g,maximum_deceleration_g,distance_km,stop_percent,acceleration_percent,deceleration_percent,cruise_percent
0,2026-08-23 10-44-49_Galeni to Kottawa.csv,14.783333,60.056272,88.0,18.170240,0.001899,0.113302,-0.056651,14.227359,5.975197,19.278467,14.768884,56.144307
1,2026-08-23 16-33-45_Kottawa to Galeni.csv,14.266667,63.763839,94.0,22.417427,0.001365,0.198278,-0.226604,14.706843,5.957944,23.714953,19.042056,48.364486
